In [32]:
%pip install mne matplotlib numpy pandas scipy scikit-learn -q

Note: you may need to restart the kernel to use updated packages.


In [33]:
import mne
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# Add the Inner Speech Dataset repository to the system path
sys.path.append(str(Path.cwd() / "Python_Processing"))

# Import custom functions from the Inner Speech Dataset repository
from lib.data_extractions import (
    extract_data_from_subject,
    load_events
)
from lib.data_processing import (
    select_time_window,
    transform_for_classificator,
)
from lib.download_helper import (
    _clone_inner_speech,
    get_derivatives,
)
from lib.utils import (
    sub_name, 
    unify_names,
)

# EEG sampling rate (256 samples per second)
fs = 256

# Select the time window of interest (in seconds)
# The cue appears at t=0, action period is between 1.5s and 3.5s
t_start = 1.5  # Start time after cue
t_end = 3.5  # End time after cue

# List with every subject (S1-S10)
N_S = list(range(1, 11))

In [ ]:
# Clone just the structure (no files yet). This step is needed to create the
data_dir = _clone_inner_speech(target_dir=Path.cwd().parent, verbose=False)
root_dir = data_dir
print(root_dir)

# Download the files needed for this script. This can take some minutes.
for i in N_S:
    get_derivatives(
        dataset_path=data_dir,
        subjects=f"{i:02d}",
        sessions=None,
        file_types=["eeg","events"],
        verbose=True,
        progress=True,
    )

[INFO] Ensuring presence of Dataset(/tmp/datalad_cache/ds003626) to get /tmp/datalad_cache/ds003626/derivatives 


/home/gustavo/Documentos/utfpr/aulas/OPT098/trabalho/ds003626
✓ Visible dataset directory ready at /home/gustavo/Documentos/utfpr/aulas/OPT098/trabalho/ds003626/ds003626
✓ Reusing cached DataLad dataset at /tmp/datalad_cache/ds003626


In [ ]:
# Loads all data from a single subject
def load_subject(root_dir: Path, n_s: int, datatype="eeg"):
    num_s = sub_name(n_s)
    x, y = [], []
    n_b_arr = [1, 2, 3]

    for n_b in n_b_arr:
        file_name = f"{root_dir}/derivatives/{num_s}/ses-0{n_b}/{num_s}_ses-0{n_b}_eeg-epo.fif"  # noqa 

        ep = mne.read_epochs(file_name, verbose="WARNING")
        x.append(ep.get_data())

        ev = pd.DataFrame(load_events(root_dir, n_s, n_b), 
                          columns=["timestamp", "direction", "condition", "session"])
        ev["subject"] = n_s
        # ev.info()

        y.append(ev)
        ch_names, times = ep.ch_names, ep.times

    X = np.concatenate(x)

    Y = pd.concat(y, ignore_index=True)
    # Y["direction"] = Y["direction"].map({0:"up", 1:"down", 2:"right", 3:"left"})
    # Y["condition"] = Y["condition"].map({0:"pronounced", 1:"inner", 2:"visualized"})

    return X, Y, ch_names, times

for i in N_S:
    X, Y, ch_names, times = load_subject(root_dir, i)
    np.save(f"S{i}_f", np.asarray(X))
    np.save(f"S{i}_y", np.asarray(Y))

    print(f"Data successfully loaded for S{i}!")
    print("\nX (EEG data) shape:", X.shape)
    print("  - Number of trials:", X.shape[0])
    print("  - Number of EEG channels:", X.shape[1])
    print("  - Number of time points per trial:", X.shape[2])
    print("  - Total duration per trial:", X.shape[2] / fs, "seconds")

    print("\nY (labels) shape:", Y.shape)
    print("  - Number of trials:", Y.shape[0])
    print("  - Label information columns:", Y.shape[1])

    # print("\nChannels: ", ch_names)
    # print("\nTimes: ", times)